In [1]:
import numpy as np
import pandas as pd
import hdbscan
import optuna
from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import euclidean_distances, haversine_distances
from sklearn.metrics import silhouette_score, silhouette_samples, davies_bouldin_score
from sklearn.impute import SimpleImputer
import warnings
import os
import glob

# --- GLOBAL SETTINGS ---
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)
RADIO_TIERRA_KM = 6371.0
np.random.seed(42)

# ------------------------------
# 1️⃣ CONFIGURACIÓN ADMINISTRATIVA
# ------------------------------
CONFIG_ADMINISTRATIVA = {
    "ARICA Y PARINACOTA": {"mcs_min": 10,  "mcs_max": 40, "ms_min": 10,  "ms_max": 40, "castigo": "duro", "limite": 1},
    "TARAPACÁ":           {"mcs_min": 15, "mcs_max": 30, "ms_min": 20, "ms_max": 40, "castigo": "suave", "limite": 2},
    "ANTOFAGASTA":        {"mcs_min": 10, "mcs_max": 20, "ms_min": 30, "ms_max": 50, "castigo": "duro", "limite": 8},
    "ATACAMA":            {"mcs_min": 25, "mcs_max": 55, "ms_min": 50, "ms_max": 90, "castigo": "duro", "limite": 5},
    "COQUIMBO":           {"mcs_min": 10, "mcs_max": 25, "ms_min": 40, "ms_max": 70, "castigo": "duro",  "limite": 3},
    "VALPARAÍSO":         {"mcs_min": 60, "mcs_max": 100, "ms_min": 60, "ms_max": 120, "castigo": "duro", "limite": 4},
    "METROPOLITANA":      {"mcs_min": 30, "mcs_max": 55, "ms_min": 50, "ms_max": 90, "castigo": "duro",  "limite": 3},
    "O'HIGGINS":          {"mcs_min": 18,  "mcs_max": 25, "ms_min": 40,  "ms_max": 55, "castigo": "duro",  "limite": 2},
    "MAULE":              {"mcs_min": 10,  "mcs_max": 20, "ms_min": 15,  "ms_max": 20, "castigo": "suave", "limite": 2},
    "DEFAULT":            {"mcs_min": 5,  "mcs_max": 15, "ms_min": 5,  "ms_max": 15, "castigo": "suave", "limite": 2}
}

MAPA_ROMANOS = {
    "ARICA Y PARINACOTA": "XV", "TARAPACÁ": "I", "ANTOFAGASTA": "II", "ATACAMA": "III",
    "COQUIMBO": "IV", "VALPARAÍSO": "V", "METROPOLITANA": "RM", "METROPOLITANA DE SANTIAGO": "RM",
    "LIBERTADOR GENERAL BERNARDO O'HIGGINS": "VI", "O'HIGGINS": "VI", "MAULE": "VII",
    "BIOBÍO": "VIII", "ARAUCANÍA": "IX", "LOS RÍOS": "XIV", "LOS LAGOS": "X",
    "AYSÉN": "XI", "MAGALLANES": "XII"
}

MAPA_ROMANOS_INV = {v:k for k,v in MAPA_ROMANOS.items()}

# ------------------------------
# 2️⃣ CARGAR DATOS Y PROCESAR
# ------------------------------
import os

FILE_CSV = "1_clustering_input_ready.csv"
FILE_NPZ = "matrices_chile_region.npz"

def smart_search_file(filename, levels_up=3):
    current_search_dir = os.getcwd()
    for i in range(levels_up + 1):
        for root, dirs, files in os.walk(current_search_dir):
            if filename in files:
                return os.path.abspath(os.path.join(root, filename))
        parent = os.path.dirname(current_search_dir)
        if parent == current_search_dir:
            break
        current_search_dir = parent
    return None

print(f"Buscando archivos cerca de: {os.getcwd()} ...")

INPUT_DATA = smart_search_file(FILE_CSV)
INPUT_MATRICES = smart_search_file(FILE_NPZ)

if not INPUT_DATA:
    raise FileNotFoundError(f"CRITICAL: No se encontró '{FILE_CSV}' buscando 3 niveles arriba.")
if not INPUT_MATRICES:
    raise FileNotFoundError(f"CRITICAL: No se encontró '{FILE_NPZ}' buscando 3 niveles arriba.")

BASE_DIR = os.path.dirname(INPUT_DATA)

print(f"Processed folder found at: {BASE_DIR}")
print(f"INPUT_MATRICES: {INPUT_MATRICES}")
print(f"INPUT_DATA: {INPUT_DATA}")

datos_npz = np.load(INPUT_MATRICES, allow_pickle=True)
print(f"►► Matriz NPZ cargada: {INPUT_MATRICES}. Regiones disponibles: {list(datos_npz.files)}")

df_raw = pd.read_csv(INPUT_DATA)
df_all = df_raw[df_raw['Estado'] == 'ACTIVA'].copy()
df_all = df_all[(df_all['RecursoPrimarioInstalacion'] == 'COBRE') | 
                (df_all['RecursoMineroInstalacion'] == 'SALMUERA (LITIO)')]
df_all = df_all.dropna(subset=['IdFaena', 'Latitud', 'Longitud', 'RegionFaena'])
df_all['IdFaena'] = df_all['IdFaena'].astype(int).astype(str)
df_all['Es_Estrategica'] = df_all['CategoriaFaena'] == 'CATEGORIA A'

def limpiar_region_simple(x):
    x = str(x).upper().replace('REGIÓN DE ', '').replace('REGIÓN DEL ', '').strip()
    return x

df_all['Region_Norm'] = df_all['RegionFaena'].apply(limpiar_region_simple)
print(f"►► Datos listos: {len(df_all)} faenas activas cargadas.")

# ------------------------------
# 3️⃣ DEFINICIÓN DE FUNCIONES
# ------------------------------
def process_region_data(region_name, matrix, ids, df_global, config, allow_single_cluster=False, top_empresas_n=5):
    try:
        ids = ids.astype(str)
        # --- JITTER & FIXES ---
        valid_mask = np.isfinite(matrix)
        max_dist = np.max(matrix[valid_mask]) if np.any(valid_mask) else 300000.0
        penalty = max_dist * 2.0
        matrix = np.nan_to_num(matrix, nan=penalty, posinf=penalty)
        matrix += np.random.uniform(0, 1e-5, matrix.shape)
        matrix = (matrix + matrix.T) / 2
        np.fill_diagonal(matrix, 0)

        common_ids = set(ids) & set(df_global['IdFaena'])
        min_req = 1 if region_name == "METROPOLITANA" else 5
        if len(common_ids) < min_req:
            return None, None

        df_region = df_global[df_global['IdFaena'].isin(common_ids)].copy()
        id_to_idx = {id_: i for i, id_ in enumerate(ids)}
        valid_indices = [id_to_idx[row_id] for row_id in df_region['IdFaena']]
        X_osrm = matrix[np.ix_(valid_indices, valid_indices)].astype(np.float64)

        # Top empresas dummies
        top_empresas = df_region['NombreEmpresa'].value_counts().head(top_empresas_n).index
        for emp in top_empresas:
            df_region[f'Empresa_{emp}'] = (df_region['NombreEmpresa'] == emp).astype(int)

        # Atributos para PCA
        cols_drop = ['RutEmpresa','NombreEmpresa','RecursoMineroInstalacion','TipoInstalacion',
                     'TipoRecursoInstalacion','RecursoPrimarioInstalacion','ComunaFaena', 
                     'NombreFaena','CategoriaFaena','IdFaena','ComunaInstalacion',
                     'NombreInstalacion','IdTipoInstalacion','IdInstalacion','Norte','Este',
                     'Huso','Datum','IdEstado','Estado','RegionFaena','RegionInstalacion', 
                     'Es_Estrategica','Region_Norm']
        dist_cols = [c for c in df_region.columns if 'dist_' in c]

        df_attr = df_region.drop(columns=[c for c in cols_drop + dist_cols if c in df_region.columns])
        df_encoded = pd.get_dummies(df_attr, columns=['ProvinciaFaena','ProvinciaInstalacion'], drop_first=True, dtype=int)
        df_numeric = df_encoded.select_dtypes(include=[np.number])
        
        # Limpieza correlacion
        corr_matrix = df_numeric.corr().abs()
        upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
        to_drop = [col for col in upper.columns if any(upper[col] > 0.995)]
        df_final_attr = df_numeric.drop(columns=to_drop)
        
        imputer = SimpleImputer(strategy='median')
        X_scaled = RobustScaler().fit_transform(imputer.fit_transform(df_final_attr))
        
        # PCA con Tracking de Importancia
        pca_obj = None
        feature_names = df_final_attr.columns
        try:
             pca_obj = PCA(n_components=0.90, random_state=42)
             X_pca = pca_obj.fit_transform(X_scaled)
             X_attr = euclidean_distances(X_pca).astype(np.float64)
        except:
             X_attr = euclidean_distances(X_scaled).astype(np.float64)

        X_osrm_norm = X_osrm / (np.max(X_osrm) if np.max(X_osrm) > 0 else 1.0)
        X_attr_norm = X_attr / (np.max(X_attr) if np.max(X_attr) > 0 else 1.0)

        # OPTUNA OPTIMIZING FOR DAVIES-BOULDIN
        def objective(trial):
            alpha = trial.suggest_float("alpha", 0.35, 0.75)
            mcs = trial.suggest_int("mcs", config["mcs_min"], config["mcs_max"])
            ms = trial.suggest_int("ms", config["ms_min"], config["ms_max"])
            
            X_comb = (alpha * X_osrm_norm) + ((1-alpha) * X_attr_norm)
            X_comb = np.ascontiguousarray(X_comb, dtype=np.float64)
            
            try:
                clusterer = hdbscan.HDBSCAN(
                    min_cluster_size=mcs,
                    min_samples=ms,
                    metric='precomputed',
                    allow_single_cluster=allow_single_cluster
                ).fit(X_comb)
                
                labels = clusterer.labels_
                n_clust = len(set(labels)) - (1 if -1 in labels else 0)
                noise_ratio = np.sum(labels==-1)/len(labels)
                
                if n_clust < 1 or noise_ratio > 0.8: return -10.0
                
                mask = labels != -1
                if len(set(labels[mask])) >= 2:
                    db_val = davies_bouldin_score(X_comb[mask], labels[mask])
                    score = -db_val 
                else:
                    return -10.0
                
                limit = config["limite"]
                if n_clust > limit:
                    if config["castigo"]=="suave": 
                        score -= (n_clust-limit) * 0.2
                    elif config["castigo"]=="duro": 
                        return -10.0
                
                return score
            except:
                return -10.0

        sampler = optuna.samplers.TPESampler(seed=42)
        study = optuna.create_study(direction="maximize", sampler=sampler)
        study.optimize(objective, n_trials=100, show_progress_bar=False)
        bp = study.best_params

        # Cluster final
        X_final = (bp['alpha']*X_osrm_norm) + ((1-bp['alpha'])*X_attr_norm)
        
        final_model = hdbscan.HDBSCAN(
            min_cluster_size=bp['mcs'],
            min_samples=bp['ms'],
            metric='precomputed',
            cluster_selection_method='eom',
            allow_single_cluster=allow_single_cluster,
            gen_min_span_tree=True 
        ).fit(X_final)

        df_region['cluster_final'] = final_model.labels_

        # --- CALCULATE FINAL SCORES ---
        try:
            dbcv_score = final_model.relative_validity_
        except:
            dbcv_score = np.nan

        # Davies-Bouldin Final
        n_clusters_final = len(set(final_model.labels_)) - (1 if -1 in final_model.labels_ else 0)
        
        if n_clusters_final > 0:
            mask_valid = final_model.labels_ != -1
            if np.sum(mask_valid) > 0 and len(set(final_model.labels_[mask_valid])) > 1:
                db_score = davies_bouldin_score(X_final[mask_valid][:, mask_valid], final_model.labels_[mask_valid])
            else:
                db_score = np.nan
        else:
            db_score = np.nan

        # --- SILHOUETTE SCORES (per sample y por cluster) ---
        mask_valid_sil = final_model.labels_ != -1
        idx_valid_sil = np.where(mask_valid_sil)[0]
        cluster_sil_dict = {}
        global_sil_score = np.nan

        if n_clusters_final >= 2 and np.sum(mask_valid_sil) >= 2:
            try:
                X_valid_sil = X_final[np.ix_(idx_valid_sil, idx_valid_sil)]
                labels_valid_sil = final_model.labels_[mask_valid_sil]
                sil_samples_arr = silhouette_samples(X_valid_sil, labels_valid_sil, metric='precomputed')
                global_sil_score = float(np.mean(sil_samples_arr))
                for cl in np.unique(labels_valid_sil):
                    cluster_sil_dict[int(cl)] = float(np.mean(sil_samples_arr[labels_valid_sil == cl]))
                sil_col = np.full(len(df_region), np.nan)
                for i, orig_idx in enumerate(idx_valid_sil):
                    sil_col[orig_idx] = sil_samples_arr[i]
                df_region['Silhouette_Sample'] = sil_col
            except Exception:
                df_region['Silhouette_Sample'] = np.nan
        else:
            df_region['Silhouette_Sample'] = np.nan

        df_region['Silhouette_Cluster_Avg'] = df_region['cluster_final'].apply(
            lambda lbl: cluster_sil_dict.get(int(lbl), np.nan) if lbl != -1 else np.nan
        )
        df_region['DB_Score_Region'] = db_score
        df_region['DBCV_Score_Region'] = dbcv_score

        # --- METADATA ---
        metadata = {
            "Region": region_name,
            "Alpha_OSRM": bp['alpha'],
            "Alpha_Attr": 1 - bp['alpha'],
            "Score_Optimized_DB": db_score,
            "Score_DBCV": dbcv_score,
            "Score_Silhouette": global_sil_score,
            "Per_Cluster_Silhouette": cluster_sil_dict,
            "Top_Features": {}
        }

        if pca_obj is not None:
            loadings = np.abs(pca_obj.components_)
            weighted_loadings = np.dot(loadings.T, pca_obj.explained_variance_ratio_)
            total_weight = np.sum(weighted_loadings)
            if total_weight > 0:
                importance_pct = (weighted_loadings / total_weight) * 100
                feat_imp = pd.Series(importance_pct, index=feature_names).sort_values(ascending=False)
                metadata["Top_Features"] = feat_imp.head(5).to_dict()

        return df_region, metadata

    except Exception as e:
        print(f"Error procesando {region_name}: {e}")
        return None, None

def rescue_noise_geo(df, matrix_times=None, percentil_rescate=30):
    df = df.copy()
    col_target = 'cluster_final'
    if col_target not in df.columns: return df
    
    labels = df[col_target].values.copy()
    idx_ruido = np.where(labels==-1)[0]
    idx_cluster = np.where(labels!=-1)[0]
    
    if len(idx_ruido) > 0 and len(idx_cluster) > 0:
        if matrix_times is not None:
            try:
                dists = matrix_times[np.ix_(idx_ruido, idx_cluster)]
            except:
                dists = haversine_distances(np.radians(df.iloc[idx_ruido][['Latitud','Longitud']].values),
                                            np.radians(df.iloc[idx_cluster][['Latitud','Longitud']].values))*RADIO_TIERRA_KM
        else:
            dists = haversine_distances(np.radians(df.iloc[idx_ruido][['Latitud','Longitud']].values),
                                        np.radians(df.iloc[idx_cluster][['Latitud','Longitud']].values))*RADIO_TIERRA_KM
        
        min_dists = np.min(dists, axis=1)
        mask_rescate = min_dists <= np.percentile(min_dists, percentil_rescate)
        nearest_indices = np.argmin(dists, axis=1)
        labels[idx_ruido[mask_rescate]] = labels[idx_cluster[nearest_indices[mask_rescate]]]
        
    df['cluster_final_v2'] = labels
    return df

# ------------------------------
# 4️⃣ EJECUCIÓN CON REPORTE DE VARIABLES
# ------------------------------
resultados_regionales = []

regiones_en_datos = df_all['Region_Norm'].unique()
MAPA_ROMANOS_INV = {v:k for k,v in MAPA_ROMANOS.items()}

for region_item in regiones_en_datos:
    if region_item in MAPA_ROMANOS:
        region_key = MAPA_ROMANOS[region_item]
        nombre_config = region_item
    elif region_item in MAPA_ROMANOS.values():
        region_key = region_item
        nombre_config = MAPA_ROMANOS_INV.get(region_item,"DEFAULT")
    else:
        continue

    print("\n" + "="*60)
    print(f"PROCESANDO: {nombre_config} (Davies-Bouldin)")
    print("="*60)

    key_matrix = f"{region_key}_matrix"
    key_ids = f"{region_key}_ids"
    key_times = f"{region_key}_times"

    if key_matrix in datos_npz and key_ids in datos_npz:
        config_key = next((k for k in CONFIG_ADMINISTRATIVA if k in nombre_config),"DEFAULT")
        config = CONFIG_ADMINISTRATIVA.get(config_key, CONFIG_ADMINISTRATIVA["DEFAULT"])
        matriz_region = datos_npz[key_matrix]
        ids_region = datos_npz[key_ids]
        times_region = datos_npz[key_times] if key_times in datos_npz else None

        allow_single = nombre_config in ["ARICA Y PARINACOTA", "METROPOLITANA", "MAULE"] or (region_key == "RM")
        top_empresas_n_use = 10 if (nombre_config == "METROPOLITANA" or region_key == "RM") else 5

        print("Iniciando Optuna...")
        
        df_res, metadata = process_region_data(
            nombre_config, 
            matriz_region, 
            ids_region, 
            df_all, 
            config,
            allow_single_cluster=allow_single,
            top_empresas_n=top_empresas_n_use
        )

        if df_res is None:
            if nombre_config == "METROPOLITANA" or region_key == "RM":
                 print("►► HDBSCAN falló en RM, creando cluster manual de todas formas.")
                 df_res = df_all[df_all['Region_Norm'].isin(["METROPOLITANA", "RM"])].copy()
                 df_res['cluster_final'] = 0
                 metadata = {"Region": "RM (Fallback)", "Alpha_OSRM": 0, "Top_Features": {},
                             "Score_Optimized_DB": np.nan, "Score_DBCV": np.nan,
                             "Score_Silhouette": np.nan, "Per_Cluster_Silhouette": {}}
            else:
                 print(f"►► No se pudo procesar {nombre_config}")
                 continue

        if metadata:
            alpha_pct = metadata.get('Alpha_OSRM', 0) * 100
            attr_pct = metadata.get('Alpha_Attr', 0) * 100
            score_db = metadata.get('Score_Optimized_DB', float('nan'))
            score_dbcv = metadata.get('Score_DBCV', float('nan'))
            score_sil = metadata.get('Score_Silhouette', float('nan'))
            per_cl_sil = metadata.get('Per_Cluster_Silhouette', {})
            db_str   = f"{score_db:.4f}"   if not np.isnan(float(score_db))   else "N/A"
            dbcv_str = f"{score_dbcv:.4f}" if not np.isnan(float(score_dbcv)) else "N/A"
            sil_str  = f"{score_sil:.4f}"  if not np.isnan(float(score_sil))  else "N/A"
            print(f"►► Importancias CLUSTERING:")
            print(f"   ► Peso Matriz Vial (OSRM): {alpha_pct:.1f}%")
            print(f"   ► Peso Atributos (Data):   {attr_pct:.1f}%")
            print(f"   ► Davies-Bouldin Score:    {db_str}  (↓ mejor)")
            print(f"   ► DBCV Score:              {dbcv_str}  (↑ mejor, rango [-1, 1])")
            print(f"   ► Silhouette Score Global: {sil_str}  (↑ mejor, rango [-1, 1])")
            print(f"   ► Variables más influyentes:")
            for feat, score in metadata.get('Top_Features', {}).items():
                print(f"      • {feat}: {score:.1f}%")

        df_proc = df_res.copy()

        if nombre_config == "METROPOLITANA" or region_key == "RM":
            df_proc = df_proc[df_proc['Region_Norm'].isin(["METROPOLITANA", "RM"])].copy()
            
            mask_merge = df_proc['cluster_final'].isin([0, 1])
            count_merge = mask_merge.sum()
            if count_merge > 0:
                print(f"►► RM Merge: Fusionando {count_merge} faenas de Cluster 0 y 1 en un solo Cluster 0.")
                df_proc.loc[mask_merge, 'cluster_final'] = 0

            print("Rescatando ruido en RM (Percentil 30)...")
            df_proc = rescue_noise_geo(df_proc, matrix_times=times_region, percentil_rescate=20)
            prefijo = "RM"
        else:
            print(f"Rescatando ruido (Percentil {30 if region_key in ['I','II','III','XV'] else 20})...")
            df_proc = rescue_noise_geo(
                df_proc, 
                matrix_times=times_region, 
                percentil_rescate=30 if region_key in ["I","II","III","XV"] else 20
            )
            prefijo = region_key if len(region_key)<=3 else region_item[:3]

        # Propagar Silhouette_Cluster_Avg a puntos rescatados del ruido (basado en cluster_final_v2)
        if metadata and metadata.get('Per_Cluster_Silhouette'):
            cluster_sil = metadata['Per_Cluster_Silhouette']
            df_proc['Silhouette_Cluster_Avg'] = df_proc['cluster_final_v2'].apply(
                lambda x: cluster_sil.get(int(x), np.nan) if int(x) != -1 else np.nan
            )

        n_iniciales = (df_proc['cluster_final'] != -1).sum()
        n_finales = (df_proc['cluster_final_v2'] != -1).sum()
        n_rescatadas = n_finales - n_iniciales
        
        print(f"   ►► DETALLE ASIGNACIÓN:")
        print(f"      ► Asignadas por Cluster (HDBSCAN): {n_iniciales}")
        print(f"      ► Rescatadas del Ruido (Geo):      {n_rescatadas}")
        print(f"      ► Total Faenas Clusterizadas:      {n_finales}")

        df_proc['Cluster_ID'] = df_proc.apply(
            lambda x: f"{prefijo}-{x['cluster_final_v2']}" if x['cluster_final_v2'] != -1 else "Ruido",
            axis=1
        )
        
        final_cluster_count = df_proc[df_proc['cluster_final_v2'] != -1]['cluster_final_v2'].nunique()
        print(f"      ► Cantidad de Clusters Finales:   {final_cluster_count}")

        resultados_regionales.append(df_proc)
        print(f"►► {nombre_config} procesada. Total minas: {len(df_proc)}")

drop_cols = [
    'Empresa_MINERA CENTINELA',
    'Empresa_MINERA ESCONDIDA LTDA.',
    'Empresa_MINERA LAS CENIZAS S.A.',
    'Empresa_MINERA SPENCE  S.A.',
    'Empresa_MINERA LOS PELAMBRES',
    'Empresa_CIA MRA. TECK CARMEN DE ANDACOLLO',
    'Empresa_MINERA BMR SPA',
    'Empresa_EL ESPINO SPA',
    'Empresa_ANGLO AMERICAN SUR S.A.',
    'Empresa_MINERA CEMIN-PULLALLI SPA',
    'Empresa_SOC. EXPLORACION Y DESARROLLO MINERO (EXPLODESA)',
    'Empresa_AURYN MINING CHILE SPA',
    'Empresa_MINERA SAN PEDRO S.A.',
    'Empresa_MINERÍA Y CONSTRUCCIÓN HAROLD MOLINA TAPIA E.I.R.L',
    'Empresa_SOCIEDAD SERVICIOS RG MINERALS SPA',
    'Empresa_MINERA ALVAREZ II SPA.',
    'Empresa_JONATHAN CUSTODIO ADASME ADASME',
    'Empresa_JUAN CARLOS FERNANDEZ SALAZAR',
    'Empresa_SANTIAGO HUMBERTO MONTENEGRO VASQUEZ',
    'Empresa_SLOGGERS MINING SPA',
    'Empresa_JUAN AUGUSTO MUNIZAGA ROMAN',
    'Empresa_ANDRES BELISARIO FELIU ESCUDERO',
    'Empresa_EXTRACCIÓN MINERA ARTESANAL OLIVARES SPA',
    'Empresa_MINERA VALLE CENTRAL S.A.',
    'Empresa_PAMPA CAMARONES SpA',
    'Empresa_YANMAFER SPA',
    'Empresa_MIGUEL ANACONA ANACONA',
    'Empresa_Sociedad Minera Santa Beatriz SpA',
    'Empresa_APOLO MINERIA SpA',
    'Empresa_CIA. MINERA DOÑA INES DE COLLAHUASI S.C.M.',
    'Empresa_CÍA. MRA. TECK QUEBRADA BLANCA S. A.',
    'Empresa_INVERSIONES GENOVA S.P.A.',
    'Empresa_CORPORACION NACIONAL DEL COBRE (CODELCO)',
    'Empresa_SOCIEDAD PUNTA DEL COBRE S.A.',
    'Empresa_MANTOVERDE S.A.',
    'Empresa_EMPRESA NACIONAL DE MINERIA (ENAMI)',
    'Empresa_MINERA LUMINA COPPER CHILE LTDA',
    # Distance and time columns to cities
    'dist_Santiago_km', 'time_Santiago_min',
    'dist_Valparaíso_km', 'time_Valparaíso_min',
    'dist_Viña_del_Mar_km', 'time_Viña_del_Mar_min',
    'dist_Antofagasta_km', 'time_Antofagasta_min',
    'dist_La_Serena_km', 'time_La_Serena_min',
    'dist_Coquimbo_km', 'time_Coquimbo_min',
    'dist_Rancagua_km', 'time_Rancagua_min',
    'dist_Talca_km', 'time_Talca_min',
    'dist_Iquique_km', 'time_Iquique_min',
    'dist_Arica_km', 'time_Arica_min',
    'dist_Calama_km', 'time_Calama_min',
    'dist_Copiapó_km', 'time_Copiapó_min',
    'dist_San_Bernardo_km', 'time_San_Bernardo_min',
    'dist_Curicó_km', 'time_Curicó_min',
    'dist_Ovalle_km', 'time_Ovalle_min',
    # Port time columns
    'Tiempo_Prt_Antofagasta', 'Tiempo_Prt_Bahia Agua Fresca',
    'Tiempo_Prt_Bahia De Valdivia', 'Tiempo_Prt_Bahia De Valparaiso',
    'Tiempo_Prt_Bahia Harris', 'Tiempo_Prt_Bahia Herradura Guayacan',
    'Tiempo_Prt_Bahia Quintero (Ventanas)', 'Tiempo_Prt_Bahia San Vicente',
    'Tiempo_Prt_Caleta Clarencia', 'Tiempo_Prt_Caleta Mina Elena',
    'Tiempo_Prt_Caleta Patillos', 'Tiempo_Prt_Coquimbo',
    'Tiempo_Prt_Coronel', 'Tiempo_Prt_Huasco',
    'Tiempo_Prt_Iquique', 'Tiempo_Prt_Lirquen',
    'Tiempo_Prt_Lota', 'Tiempo_Prt_Mejillones',
    'Tiempo_Prt_Penco', 'Tiempo_Prt_Port San Juan De La Juan',
    'Tiempo_Prt_Puerto Caldera', 'Tiempo_Prt_Puerto Calderilla',
    'Tiempo_Prt_Puerto Castro', 'Tiempo_Prt_Puerto Chacabuco',
    'Tiempo_Prt_Puerto Chanaral', 'Tiempo_Prt_Puerto De Corral',
    'Tiempo_Prt_Puerto Montt', 'Tiempo_Prt_Puerto Natales',
    'Tiempo_Prt_Puerto Quemchi', 'Tiempo_Prt_Puerto San Antonio',
    'Tiempo_Prt_Puerto Sara', 'Tiempo_Prt_Puerto Williams',
    'Tiempo_Prt_Puerto Yartou', 'Tiempo_Prt_Punta Delgada',
    'Tiempo_Prt_Rada De Arica', 'Tiempo_Prt_Rada Punta Arenas',
    'Tiempo_Prt_Talcahuano', 'Tiempo_Prt_Taltal',
    'Tiempo_Prt_Tocopilla',
    # Clustering and region columns
    'Es_Estrategica'
]

# ------------------------------
# Guardar resultados
# ------------------------------
if resultados_regionales:
    df_final = pd.concat(resultados_regionales, ignore_index=True)
    
    df_final = df_final.drop(columns=[col for col in drop_cols if col in df_final.columns])
    
    # ------------------------------
    # MANUAL CLUSTER REASSIGNMENT
    # ------------------------------
    mask_lomas_bayas = (df_final['NombreEmpresa'] == 'COMPAÑÍA MINERA  LOMAS BAYAS') & (df_final['Cluster_ID'] == 'II-2')
    n_reassigned = mask_lomas_bayas.sum()
    
    if n_reassigned > 0:
        df_final.loc[mask_lomas_bayas, 'cluster_final_v2'] = 3
        df_final.loc[mask_lomas_bayas, 'Cluster_ID'] = 'II-3'
        print(f"\n►► REASIGNACIÓN MANUAL:")
        print(f"   ► {n_reassigned} faena(s) de COMPAÑÍA MINERA  LOMAS BAYAS movidas de II-2 a II-3")
    else:
        print(f"\n►► ADVERTENCIA: No se encontraron faenas de COMPAÑÍA MINERA  LOMAS BAYAS en cluster II-2 para reasignar")
    
    print(f"\n►► PROCESO FINALIZADO. Total Minas: {len(df_final)}")
    
    OUTPUT_FILE = os.path.join(BASE_DIR,"2_regional_davbou_hdbs.csv")
    df_final.to_csv(OUTPUT_FILE, index=False)
    print(f"► Resultados guardados en: {OUTPUT_FILE}")
else:
    print("\n►► No se generaron resultados.")


Buscando archivos cerca de: /Users/mac/TrabajoTesis/FinalResultsFolder/02_Clustering/notebooks ...
Processed folder found at: /Users/mac/TrabajoTesis/FinalResultsFolder/02_Clustering/outputs
INPUT_MATRICES: /Users/mac/TrabajoTesis/FinalResultsFolder/02_Clustering/outputs/matrices_chile_region.npz
INPUT_DATA: /Users/mac/TrabajoTesis/FinalResultsFolder/02_Clustering/outputs/1_clustering_input_ready.csv
►► Matriz NPZ cargada: /Users/mac/TrabajoTesis/FinalResultsFolder/02_Clustering/outputs/matrices_chile_region.npz. Regiones disponibles: ['I_matrix', 'I_ids', 'II_matrix', 'II_ids', 'III_matrix', 'III_ids', 'IV_matrix', 'IV_ids', 'RM_matrix', 'RM_ids', 'V_matrix', 'V_ids', 'VI_matrix', 'VI_ids', 'VII_matrix', 'VII_ids', 'XV_matrix', 'XV_ids']
►► Datos listos: 7930 faenas activas cargadas.

PROCESANDO: ATACAMA (Davies-Bouldin)
Iniciando Optuna...


►► Importancias CLUSTERING:
   ► Peso Matriz Vial (OSRM): 46.4%
   ► Peso Atributos (Data):   53.6%
   ► Davies-Bouldin Score:    0.3017  (↓ mejor)
   ► DBCV Score:              N/A  (↑ mejor, rango [-1, 1])
   ► Silhouette Score Global: 0.5809  (↑ mejor, rango [-1, 1])
   ► Variables más influyentes:
      • Cota: 9.5%
      • Longitud: 8.8%
      • Tiempo_Prt_Puerto Chanaral: 8.8%
      • Tiempo_Prt_Puerto Caldera: 8.6%
      • time_Antofagasta_min: 8.3%
Rescatando ruido (Percentil 30)...
   ►► DETALLE ASIGNACIÓN:
      ► Asignadas por Cluster (HDBSCAN): 1301
      ► Rescatadas del Ruido (Geo):      322
      ► Total Faenas Clusterizadas:      1623
      ► Cantidad de Clusters Finales:   4
►► ATACAMA procesada. Total minas: 2372

PROCESANDO: ANTOFAGASTA (Davies-Bouldin)
Iniciando Optuna...


►► Importancias CLUSTERING:
   ► Peso Matriz Vial (OSRM): 49.4%
   ► Peso Atributos (Data):   50.6%
   ► Davies-Bouldin Score:    0.3567  (↓ mejor)
   ► DBCV Score:              N/A  (↑ mejor, rango [-1, 1])
   ► Silhouette Score Global: 0.6806  (↑ mejor, rango [-1, 1])
   ► Variables más influyentes:
      • time_Antofagasta_min: 9.4%
      • Tiempo_Prt_Mejillones: 9.1%
      • Tiempo_Prt_Bahia Agua Fresca: 8.0%
      • Tiempo_Prt_Taltal: 7.6%
      • time_Santiago_min: 7.5%
Rescatando ruido (Percentil 30)...
   ►► DETALLE ASIGNACIÓN:
      ► Asignadas por Cluster (HDBSCAN): 886
      ► Rescatadas del Ruido (Geo):      170
      ► Total Faenas Clusterizadas:      1056
      ► Cantidad de Clusters Finales:   7
►► ANTOFAGASTA procesada. Total minas: 1453

PROCESANDO: COQUIMBO (Davies-Bouldin)
Iniciando Optuna...


►► Importancias CLUSTERING:
   ► Peso Matriz Vial (OSRM): 47.0%
   ► Peso Atributos (Data):   53.0%
   ► Davies-Bouldin Score:    0.4731  (↓ mejor)
   ► DBCV Score:              N/A  (↑ mejor, rango [-1, 1])
   ► Silhouette Score Global: 0.5494  (↑ mejor, rango [-1, 1])
   ► Variables más influyentes:
      • Cota: 14.0%
      • time_Antofagasta_min: 10.7%
      • Longitud: 10.3%
      • time_Santiago_min: 10.0%
      • time_La_Serena_min: 10.0%
Rescatando ruido (Percentil 20)...
   ►► DETALLE ASIGNACIÓN:
      ► Asignadas por Cluster (HDBSCAN): 906
      ► Rescatadas del Ruido (Geo):      173
      ► Total Faenas Clusterizadas:      1079
      ► Cantidad de Clusters Finales:   3
►► COQUIMBO procesada. Total minas: 1768

PROCESANDO: VALPARAÍSO (Davies-Bouldin)
Iniciando Optuna...


►► Importancias CLUSTERING:
   ► Peso Matriz Vial (OSRM): 75.0%
   ► Peso Atributos (Data):   25.0%
   ► Davies-Bouldin Score:    0.3379  (↓ mejor)
   ► DBCV Score:              N/A  (↑ mejor, rango [-1, 1])
   ► Silhouette Score Global: 0.5929  (↑ mejor, rango [-1, 1])
   ► Variables más influyentes:
      • Longitud: 9.5%
      • time_Santiago_min: 9.2%
      • time_Antofagasta_min: 9.0%
      • Tiempo_Prt_Puerto San Antonio: 8.9%
      • Latitud: 8.5%
Rescatando ruido (Percentil 20)...
   ►► DETALLE ASIGNACIÓN:
      ► Asignadas por Cluster (HDBSCAN): 1405
      ► Rescatadas del Ruido (Geo):      15
      ► Total Faenas Clusterizadas:      1420
      ► Cantidad de Clusters Finales:   2
►► VALPARAÍSO procesada. Total minas: 1480

PROCESANDO: METROPOLITANA DE SANTIAGO (Davies-Bouldin)
Iniciando Optuna...


►► Importancias CLUSTERING:
   ► Peso Matriz Vial (OSRM): 57.2%
   ► Peso Atributos (Data):   42.8%
   ► Davies-Bouldin Score:    0.1678  (↓ mejor)
   ► DBCV Score:              N/A  (↑ mejor, rango [-1, 1])
   ► Silhouette Score Global: 0.8734  (↑ mejor, rango [-1, 1])
   ► Variables más influyentes:
      • Latitud: 70.7%
      • time_Santiago_min: 8.2%
      • time_Valparaíso_min: 6.4%
      • time_Antofagasta_min: 4.3%
      • Tiempo_Prt_Taltal: 2.9%
►► RM Merge: Fusionando 205 faenas de Cluster 0 y 1 en un solo Cluster 0.
Rescatando ruido en RM (Percentil 30)...
   ►► DETALLE ASIGNACIÓN:
      ► Asignadas por Cluster (HDBSCAN): 205
      ► Rescatadas del Ruido (Geo):      39
      ► Total Faenas Clusterizadas:      244
      ► Cantidad de Clusters Finales:   1
►► METROPOLITANA DE SANTIAGO procesada. Total minas: 400

PROCESANDO: O'HIGGINS (Davies-Bouldin)
Iniciando Optuna...


►► Importancias CLUSTERING:
   ► Peso Matriz Vial (OSRM): 60.2%
   ► Peso Atributos (Data):   39.8%
   ► Davies-Bouldin Score:    0.0849  (↓ mejor)
   ► DBCV Score:              N/A  (↑ mejor, rango [-1, 1])
   ► Silhouette Score Global: 0.9123  (↑ mejor, rango [-1, 1])
   ► Variables más influyentes:
      • Longitud: 57.4%
      • Cota: 17.3%
      • Empresa_JUAN AUGUSTO MUNIZAGA ROMAN: 4.6%
      • time_Talca_min: 4.0%
      • time_Santiago_min: 4.0%
Rescatando ruido (Percentil 20)...
   ►► DETALLE ASIGNACIÓN:
      ► Asignadas por Cluster (HDBSCAN): 123
      ► Rescatadas del Ruido (Geo):      27
      ► Total Faenas Clusterizadas:      150
      ► Cantidad de Clusters Finales:   2
►► O'HIGGINS procesada. Total minas: 256

PROCESANDO: ARICA Y PARINACOTA (Davies-Bouldin)
Iniciando Optuna...


►► Importancias CLUSTERING:
   ► Peso Matriz Vial (OSRM): 50.0%
   ► Peso Atributos (Data):   50.0%
   ► Davies-Bouldin Score:    N/A  (↓ mejor)
   ► DBCV Score:              N/A  (↑ mejor, rango [-1, 1])
   ► Silhouette Score Global: N/A  (↑ mejor, rango [-1, 1])
   ► Variables más influyentes:
      • Cota: 46.9%
      • Longitud: 18.1%
      • time_Santiago_min: 16.9%
      • time_Arica_min: 8.9%
      • Latitud: 5.9%
Rescatando ruido (Percentil 30)...
   ►► DETALLE ASIGNACIÓN:
      ► Asignadas por Cluster (HDBSCAN): 40
      ► Rescatadas del Ruido (Geo):      3
      ► Total Faenas Clusterizadas:      43
      ► Cantidad de Clusters Finales:   1
►► ARICA Y PARINACOTA procesada. Total minas: 50

PROCESANDO: MAULE (Davies-Bouldin)
Iniciando Optuna...
►► No se pudo procesar MAULE

PROCESANDO: TARAPACÁ (Davies-Bouldin)
Iniciando Optuna...


►► Importancias CLUSTERING:
   ► Peso Matriz Vial (OSRM): 75.0%
   ► Peso Atributos (Data):   25.0%
   ► Davies-Bouldin Score:    0.1687  (↓ mejor)
   ► DBCV Score:              N/A  (↑ mejor, rango [-1, 1])
   ► Silhouette Score Global: 0.8658  (↑ mejor, rango [-1, 1])
   ► Variables más influyentes:
      • time_Arica_min: 12.4%
      • time_Iquique_min: 9.5%
      • Tiempo_Prt_Mejillones: 7.8%
      • Tiempo_Prt_Tocopilla: 7.6%
      • time_Santiago_min: 7.6%
Rescatando ruido (Percentil 30)...
   ►► DETALLE ASIGNACIÓN:
      ► Asignadas por Cluster (HDBSCAN): 78
      ► Rescatadas del Ruido (Geo):      22
      ► Total Faenas Clusterizadas:      100
      ► Cantidad de Clusters Finales:   2
►► TARAPACÁ procesada. Total minas: 149

►► REASIGNACIÓN MANUAL:
   ► 20 faena(s) de COMPAÑÍA MINERA  LOMAS BAYAS movidas de II-2 a II-3

►► PROCESO FINALIZADO. Total Minas: 7928
► Resultados guardados en: /Users/mac/TrabajoTesis/FinalResultsFolder/02_Clustering/outputs/2_regional_davbou_hdbs.csv

In [2]:
import numpy as np
import pandas as pd
import hdbscan
import optuna
from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import euclidean_distances, haversine_distances
from sklearn.metrics import silhouette_score, silhouette_samples, davies_bouldin_score
from sklearn.impute import SimpleImputer
import warnings
import os
import glob

# --- GLOBAL SETTINGS ---
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)
RADIO_TIERRA_KM = 6371.0
np.random.seed(42)

# ------------------------------
# 1️⃣ CONFIGURACIÓN ADMINISTRATIVA
# ------------------------------
CONFIG_ADMINISTRATIVA = {
    "ARICA Y PARINACOTA": {"mcs_min": 10,  "mcs_max": 40, "ms_min": 10,  "ms_max": 40, "castigo": "duro", "limite": 1},
    "TARAPACÁ":           {"mcs_min": 15, "mcs_max": 30, "ms_min": 20, "ms_max": 40, "castigo": "suave", "limite": 2},
    "ANTOFAGASTA":        {"mcs_min": 10, "mcs_max": 20, "ms_min": 30, "ms_max": 50, "castigo": "duro", "limite": 8},
    "ATACAMA":            {"mcs_min": 25, "mcs_max": 55, "ms_min": 50, "ms_max": 90, "castigo": "duro", "limite": 5},
    "COQUIMBO":           {"mcs_min": 10, "mcs_max": 25, "ms_min": 40, "ms_max": 70, "castigo": "duro",  "limite": 3},
    "VALPARAÍSO":         {"mcs_min": 60, "mcs_max": 100, "ms_min": 60, "ms_max": 120, "castigo": "duro", "limite": 4},
    "METROPOLITANA":      {"mcs_min": 30, "mcs_max": 55, "ms_min": 50, "ms_max": 90, "castigo": "duro",  "limite": 3},
    "O'HIGGINS":          {"mcs_min": 18,  "mcs_max": 25, "ms_min": 40,  "ms_max": 55, "castigo": "duro",  "limite": 2},
    "MAULE":              {"mcs_min": 10,  "mcs_max": 20, "ms_min": 15,  "ms_max": 20, "castigo": "suave", "limite": 2},
    "DEFAULT":            {"mcs_min": 5,  "mcs_max": 15, "ms_min": 5,  "ms_max": 15, "castigo": "suave", "limite": 2}
}

MAPA_ROMANOS = {
    "ARICA Y PARINACOTA": "XV", "TARAPACÁ": "I", "ANTOFAGASTA": "II", "ATACAMA": "III",
    "COQUIMBO": "IV", "VALPARAÍSO": "V", "METROPOLITANA": "RM", "METROPOLITANA DE SANTIAGO": "RM",
    "LIBERTADOR GENERAL BERNARDO O'HIGGINS": "VI", "O'HIGGINS": "VI", "MAULE": "VII",
    "BIOBÍO": "VIII", "ARAUCANÍA": "IX", "LOS RÍOS": "XIV", "LOS LAGOS": "X",
    "AYSÉN": "XI", "MAGALLANES": "XII"
}

MAPA_ROMANOS_INV = {v:k for k,v in MAPA_ROMANOS.items()}

# ------------------------------
# 2️⃣ CARGAR DATOS Y PROCESAR
# ------------------------------
import os

FILE_CSV = "1_clustering_input_ready.csv"
FILE_NPZ = "matrices_chile_region.npz"

def smart_search_file(filename, levels_up=3):
    current_search_dir = os.getcwd()
    for i in range(levels_up + 1):
        for root, dirs, files in os.walk(current_search_dir):
            if filename in files:
                return os.path.abspath(os.path.join(root, filename))
        parent = os.path.dirname(current_search_dir)
        if parent == current_search_dir:
            break
        current_search_dir = parent
    return None

print(f"Buscando archivos cerca de: {os.getcwd()} ...")

INPUT_DATA = smart_search_file(FILE_CSV)
INPUT_MATRICES = smart_search_file(FILE_NPZ)

if not INPUT_DATA:
    raise FileNotFoundError(f"CRITICAL: No se encontró '{FILE_CSV}' buscando 3 niveles arriba.")
if not INPUT_MATRICES:
    raise FileNotFoundError(f"CRITICAL: No se encontró '{FILE_NPZ}' buscando 3 niveles arriba.")

BASE_DIR = os.path.dirname(INPUT_DATA)

print(f"Processed folder found at: {BASE_DIR}")
print(f"INPUT_MATRICES: {INPUT_MATRICES}")
print(f"INPUT_DATA: {INPUT_DATA}")

datos_npz = np.load(INPUT_MATRICES, allow_pickle=True)
print(f"►► Matriz NPZ cargada: {INPUT_MATRICES}. Regiones disponibles: {list(datos_npz.files)}")

df_raw = pd.read_csv(INPUT_DATA)
df_all = df_raw[df_raw['Estado'] == 'ACTIVA'].copy()
df_all = df_all[(df_all['RecursoPrimarioInstalacion'] == 'COBRE') | 
                (df_all['RecursoMineroInstalacion'] == 'SALMUERA (LITIO)')]
df_all = df_all.dropna(subset=['IdFaena', 'Latitud', 'Longitud', 'RegionFaena'])
df_all['IdFaena'] = df_all['IdFaena'].astype(int).astype(str)
df_all['Es_Estrategica'] = df_all['CategoriaFaena'] == 'CATEGORIA A'

def limpiar_region_simple(x):
    x = str(x).upper().replace('REGIÓN DE ', '').replace('REGIÓN DEL ', '').strip()
    return x

df_all['Region_Norm'] = df_all['RegionFaena'].apply(limpiar_region_simple)
print(f"►► Datos listos: {len(df_all)} faenas activas cargadas.")

# ------------------------------
# 3️⃣ DEFINICIÓN DE FUNCIONES
# ------------------------------
def process_region_data(region_name, matrix, ids, df_global, config, allow_single_cluster=False, top_empresas_n=5):
    try:
        ids = ids.astype(str)
        # --- JITTER & FIXES ---
        valid_mask = np.isfinite(matrix)
        max_dist = np.max(matrix[valid_mask]) if np.any(valid_mask) else 300000.0
        penalty = max_dist * 2.0
        matrix = np.nan_to_num(matrix, nan=penalty, posinf=penalty)
        matrix += np.random.uniform(0, 1e-5, matrix.shape)
        matrix = (matrix + matrix.T) / 2
        np.fill_diagonal(matrix, 0)

        common_ids = set(ids) & set(df_global['IdFaena'])
        min_req = 1 if region_name == "METROPOLITANA" else 5
        if len(common_ids) < min_req:
            return None, None

        df_region = df_global[df_global['IdFaena'].isin(common_ids)].copy()
        id_to_idx = {id_: i for i, id_ in enumerate(ids)}
        valid_indices = [id_to_idx[row_id] for row_id in df_region['IdFaena']]
        X_osrm = matrix[np.ix_(valid_indices, valid_indices)].astype(np.float64)

        # Top empresas dummies
        top_empresas = df_region['NombreEmpresa'].value_counts().head(top_empresas_n).index
        for emp in top_empresas:
            df_region[f'Empresa_{emp}'] = (df_region['NombreEmpresa'] == emp).astype(int)

        # Atributos para PCA
        cols_drop = ['RutEmpresa','NombreEmpresa','RecursoMineroInstalacion','TipoInstalacion',
                     'TipoRecursoInstalacion','RecursoPrimarioInstalacion','ComunaFaena', 
                     'NombreFaena','CategoriaFaena','IdFaena','ComunaInstalacion',
                     'NombreInstalacion','IdTipoInstalacion','IdInstalacion','Norte','Este',
                     'Huso','Datum','IdEstado','Estado','RegionFaena','RegionInstalacion', 
                     'Es_Estrategica','Region_Norm']
        dist_cols = [c for c in df_region.columns if 'dist_' in c]

        df_attr = df_region.drop(columns=[c for c in cols_drop + dist_cols if c in df_region.columns])
        df_encoded = pd.get_dummies(df_attr, columns=['ProvinciaFaena','ProvinciaInstalacion'], drop_first=True, dtype=int)
        df_numeric = df_encoded.select_dtypes(include=[np.number])
        
        # Limpieza correlacion
        corr_matrix = df_numeric.corr().abs()
        upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
        to_drop = [col for col in upper.columns if any(upper[col] > 0.995)]
        df_final_attr = df_numeric.drop(columns=to_drop)
        
        imputer = SimpleImputer(strategy='median')
        X_scaled = RobustScaler().fit_transform(imputer.fit_transform(df_final_attr))
        
        # PCA con Tracking de Importancia
        pca_obj = None
        feature_names = df_final_attr.columns
        try:
             pca_obj = PCA(n_components=0.90, random_state=42)
             X_pca = pca_obj.fit_transform(X_scaled)
             X_attr = euclidean_distances(X_pca).astype(np.float64)
        except:
             X_attr = euclidean_distances(X_scaled).astype(np.float64)

        X_osrm_norm = X_osrm / (np.max(X_osrm) if np.max(X_osrm) > 0 else 1.0)
        X_attr_norm = X_attr / (np.max(X_attr) if np.max(X_attr) > 0 else 1.0)

        # OPTUNA OPTIMIZING FOR DAVIES-BOULDIN
        def objective(trial):
            alpha = trial.suggest_float("alpha", 0.35, 0.75)
            mcs = trial.suggest_int("mcs", config["mcs_min"], config["mcs_max"])
            ms = trial.suggest_int("ms", config["ms_min"], config["ms_max"])
            
            X_comb = (alpha * X_osrm_norm) + ((1-alpha) * X_attr_norm)
            X_comb = np.ascontiguousarray(X_comb, dtype=np.float64)
            
            try:
                clusterer = hdbscan.HDBSCAN(
                    min_cluster_size=mcs,
                    min_samples=ms,
                    metric='precomputed',
                    allow_single_cluster=allow_single_cluster
                ).fit(X_comb)
                
                labels = clusterer.labels_
                n_clust = len(set(labels)) - (1 if -1 in labels else 0)
                noise_ratio = np.sum(labels==-1)/len(labels)
                
                if n_clust < 1 or noise_ratio > 0.8: return -10.0
                
                mask = labels != -1
                if len(set(labels[mask])) >= 2:
                    db_val = davies_bouldin_score(X_comb[mask], labels[mask])
                    score = -db_val 
                else:
                    return -10.0
                
                limit = config["limite"]
                if n_clust > limit:
                    if config["castigo"]=="suave": 
                        score -= (n_clust-limit) * 0.2
                    elif config["castigo"]=="duro": 
                        return -10.0
                
                return score
            except:
                return -10.0

        sampler = optuna.samplers.TPESampler(seed=42)
        study = optuna.create_study(direction="maximize", sampler=sampler)
        study.optimize(objective, n_trials=100, show_progress_bar=False)
        bp = study.best_params

        # Cluster final
        X_final = (bp['alpha']*X_osrm_norm) + ((1-bp['alpha'])*X_attr_norm)
        
        final_model = hdbscan.HDBSCAN(
            min_cluster_size=bp['mcs'],
            min_samples=bp['ms'],
            metric='precomputed',
            cluster_selection_method='eom',
            allow_single_cluster=allow_single_cluster,
            gen_min_span_tree=True 
        ).fit(X_final)

        df_region['cluster_final'] = final_model.labels_

        # --- CALCULATE FINAL SCORES ---
        def compute_dbcv_precomputed(dist_matrix, labels):
            """
            DBCV (Moulavi et al. 2014) sobre matriz de distancias precomputada.
            Retorna score en [-1, 1] (↑ mejor). No altera la lógica de clustering.
            """
            try:
                from scipy.sparse.csgraph import minimum_spanning_tree
                from scipy.sparse import csr_matrix

                unique_labels = [l for l in np.unique(labels) if l != -1]
                if len(unique_labels) < 2:
                    return np.nan

                n = len(labels)

                # 1) Core distance: vecino más cercano dentro del mismo cluster
                core_dists = np.zeros(n)
                for lbl in unique_labels:
                    idx_cl = np.where(labels == lbl)[0]
                    if len(idx_cl) < 2:
                        continue
                    sub = dist_matrix[np.ix_(idx_cl, idx_cl)].copy()
                    np.fill_diagonal(sub, np.inf)
                    core_dists[idx_cl] = np.min(sub, axis=1)

                # 2) Mutual Reachability Distance: mrd(a,b) = max(core(a), core(b), dist(a,b))
                mrd = np.maximum(np.maximum(core_dists[:, None], core_dists[None, :]), dist_matrix)

                # 3) Por cluster: MST sobre MRD → DSC (densidad intra) y DSPC (separación inter)
                cluster_scores, cluster_sizes = [], []
                for lbl in unique_labels:
                    idx_cl = np.where(labels == lbl)[0]
                    if len(idx_cl) < 2:
                        cluster_scores.append(0.0)
                        cluster_sizes.append(len(idx_cl))
                        continue

                    sub_mrd = mrd[np.ix_(idx_cl, idx_cl)]
                    mst_arr = minimum_spanning_tree(csr_matrix(sub_mrd)).toarray()
                    edges = mst_arr[mst_arr > 0]
                    if len(edges) == 0:
                        cluster_scores.append(0.0)
                        cluster_sizes.append(len(idx_cl))
                        continue

                    dsc = np.max(edges)  # Density Sparseness of Cluster

                    dspc_vals = [
                        np.min(mrd[np.ix_(idx_cl, np.where(labels == lbl2)[0])])
                        for lbl2 in unique_labels if lbl2 != lbl
                    ]
                    dspc = min(dspc_vals) if dspc_vals else 0.0

                    vi = (dspc - dsc) / max(dspc, dsc) if max(dspc, dsc) > 0 else 0.0
                    cluster_scores.append(vi)
                    cluster_sizes.append(len(idx_cl))

                # 4) Score global ponderado por tamaño de cluster
                total = sum(cluster_sizes)
                return float(sum(s * sz for s, sz in zip(cluster_scores, cluster_sizes)) / total) if total > 0 else np.nan

            except Exception:
                return np.nan

        # DBCV calculado sobre X_osrm (distancias viales reales en km).
        # Solo es métrica de reporte — no afecta Optuna ni la selección de clusters.
        try:
            dbcv_score = compute_dbcv_precomputed(X_osrm, final_model.labels_)
        except Exception:
            dbcv_score = np.nan

        # Davies-Bouldin Final
        n_clusters_final = len(set(final_model.labels_)) - (1 if -1 in final_model.labels_ else 0)
        
        if n_clusters_final > 0:
            mask_valid = final_model.labels_ != -1
            if np.sum(mask_valid) > 0 and len(set(final_model.labels_[mask_valid])) > 1:
                db_score = davies_bouldin_score(X_final[mask_valid][:, mask_valid], final_model.labels_[mask_valid])
            else:
                db_score = np.nan
        else:
            db_score = np.nan

        # --- SILHOUETTE SCORES (per sample y por cluster) ---
        mask_valid_sil = final_model.labels_ != -1
        idx_valid_sil = np.where(mask_valid_sil)[0]
        cluster_sil_dict = {}
        global_sil_score = np.nan

        if n_clusters_final >= 2 and np.sum(mask_valid_sil) >= 2:
            try:
                X_valid_sil = X_final[np.ix_(idx_valid_sil, idx_valid_sil)]
                labels_valid_sil = final_model.labels_[mask_valid_sil]
                sil_samples_arr = silhouette_samples(X_valid_sil, labels_valid_sil, metric='precomputed')
                global_sil_score = float(np.mean(sil_samples_arr))
                for cl in np.unique(labels_valid_sil):
                    cluster_sil_dict[int(cl)] = float(np.mean(sil_samples_arr[labels_valid_sil == cl]))
                sil_col = np.full(len(df_region), np.nan)
                for i, orig_idx in enumerate(idx_valid_sil):
                    sil_col[orig_idx] = sil_samples_arr[i]
                df_region['Silhouette_Sample'] = sil_col
            except Exception:
                df_region['Silhouette_Sample'] = np.nan
        else:
            df_region['Silhouette_Sample'] = np.nan

        df_region['Silhouette_Cluster_Avg'] = df_region['cluster_final'].apply(
            lambda lbl: cluster_sil_dict.get(int(lbl), np.nan) if lbl != -1 else np.nan
        )
        df_region['DB_Score_Region'] = db_score
        df_region['DBCV_Score_Region'] = dbcv_score

        # --- METADATA ---
        metadata = {
            "Region": region_name,
            "Alpha_OSRM": bp['alpha'],
            "Alpha_Attr": 1 - bp['alpha'],
            "Score_Optimized_DB": db_score,
            "Score_DBCV": dbcv_score,
            "Score_Silhouette": global_sil_score,
            "Per_Cluster_Silhouette": cluster_sil_dict,
            "Top_Features": {}
        }

        if pca_obj is not None:
            loadings = np.abs(pca_obj.components_)
            weighted_loadings = np.dot(loadings.T, pca_obj.explained_variance_ratio_)
            total_weight = np.sum(weighted_loadings)
            if total_weight > 0:
                importance_pct = (weighted_loadings / total_weight) * 100
                feat_imp = pd.Series(importance_pct, index=feature_names).sort_values(ascending=False)
                metadata["Top_Features"] = feat_imp.head(5).to_dict()

        return df_region, metadata

    except Exception as e:
        print(f"Error procesando {region_name}: {e}")
        return None, None

def rescue_noise_geo(df, matrix_times=None, percentil_rescate=30):
    df = df.copy()
    col_target = 'cluster_final'
    if col_target not in df.columns: return df
    
    labels = df[col_target].values.copy()
    idx_ruido = np.where(labels==-1)[0]
    idx_cluster = np.where(labels!=-1)[0]
    
    if len(idx_ruido) > 0 and len(idx_cluster) > 0:
        if matrix_times is not None:
            try:
                dists = matrix_times[np.ix_(idx_ruido, idx_cluster)]
            except:
                dists = haversine_distances(np.radians(df.iloc[idx_ruido][['Latitud','Longitud']].values),
                                            np.radians(df.iloc[idx_cluster][['Latitud','Longitud']].values))*RADIO_TIERRA_KM
        else:
            dists = haversine_distances(np.radians(df.iloc[idx_ruido][['Latitud','Longitud']].values),
                                        np.radians(df.iloc[idx_cluster][['Latitud','Longitud']].values))*RADIO_TIERRA_KM
        
        min_dists = np.min(dists, axis=1)
        mask_rescate = min_dists <= np.percentile(min_dists, percentil_rescate)
        nearest_indices = np.argmin(dists, axis=1)
        labels[idx_ruido[mask_rescate]] = labels[idx_cluster[nearest_indices[mask_rescate]]]
        
    df['cluster_final_v2'] = labels
    return df

# ------------------------------
# 4️⃣ EJECUCIÓN CON REPORTE DE VARIABLES
# ------------------------------
resultados_regionales = []

regiones_en_datos = df_all['Region_Norm'].unique()
MAPA_ROMANOS_INV = {v:k for k,v in MAPA_ROMANOS.items()}

for region_item in regiones_en_datos:
    if region_item in MAPA_ROMANOS:
        region_key = MAPA_ROMANOS[region_item]
        nombre_config = region_item
    elif region_item in MAPA_ROMANOS.values():
        region_key = region_item
        nombre_config = MAPA_ROMANOS_INV.get(region_item,"DEFAULT")
    else:
        continue

    print("\n" + "="*60)
    print(f"PROCESANDO: {nombre_config} (Davies-Bouldin)")
    print("="*60)

    key_matrix = f"{region_key}_matrix"
    key_ids = f"{region_key}_ids"
    key_times = f"{region_key}_times"

    if key_matrix in datos_npz and key_ids in datos_npz:
        config_key = next((k for k in CONFIG_ADMINISTRATIVA if k in nombre_config),"DEFAULT")
        config = CONFIG_ADMINISTRATIVA.get(config_key, CONFIG_ADMINISTRATIVA["DEFAULT"])
        matriz_region = datos_npz[key_matrix]
        ids_region = datos_npz[key_ids]
        times_region = datos_npz[key_times] if key_times in datos_npz else None

        allow_single = nombre_config in ["ARICA Y PARINACOTA", "METROPOLITANA", "MAULE"] or (region_key == "RM")
        top_empresas_n_use = 10 if (nombre_config == "METROPOLITANA" or region_key == "RM") else 5

        print("Iniciando Optuna...")
        
        df_res, metadata = process_region_data(
            nombre_config, 
            matriz_region, 
            ids_region, 
            df_all, 
            config,
            allow_single_cluster=allow_single,
            top_empresas_n=top_empresas_n_use
        )

        if df_res is None:
            if nombre_config == "METROPOLITANA" or region_key == "RM":
                 print("►► HDBSCAN falló en RM, creando cluster manual de todas formas.")
                 df_res = df_all[df_all['Region_Norm'].isin(["METROPOLITANA", "RM"])].copy()
                 df_res['cluster_final'] = 0
                 metadata = {"Region": "RM (Fallback)", "Alpha_OSRM": 0, "Top_Features": {},
                             "Score_Optimized_DB": np.nan, "Score_DBCV": np.nan,
                             "Score_Silhouette": np.nan, "Per_Cluster_Silhouette": {}}
            else:
                 print(f"►► No se pudo procesar {nombre_config}")
                 continue

        if metadata:
            alpha_pct = metadata.get('Alpha_OSRM', 0) * 100
            attr_pct = metadata.get('Alpha_Attr', 0) * 100
            score_db = metadata.get('Score_Optimized_DB', float('nan'))
            score_dbcv = metadata.get('Score_DBCV', float('nan'))
            score_sil = metadata.get('Score_Silhouette', float('nan'))
            per_cl_sil = metadata.get('Per_Cluster_Silhouette', {})
            db_str   = f"{score_db:.4f}"   if not np.isnan(float(score_db))   else "N/A"
            dbcv_str = f"{score_dbcv:.4f}" if not np.isnan(float(score_dbcv)) else "N/A"
            sil_str  = f"{score_sil:.4f}"  if not np.isnan(float(score_sil))  else "N/A"
            print(f"►► Importancias CLUSTERING:")
            print(f"   ► Peso Matriz Vial (OSRM): {alpha_pct:.1f}%")
            print(f"   ► Peso Atributos (Data):   {attr_pct:.1f}%")
            print(f"   ► Davies-Bouldin Score:    {db_str}  (↓ mejor)")
            print(f"   ► DBCV Score:              {dbcv_str}  (↑ mejor, rango [-1, 1])")
            print(f"   ► Silhouette Score Global: {sil_str}  (↑ mejor, rango [-1, 1])")
            print(f"   ► Variables más influyentes:")
            for feat, score in metadata.get('Top_Features', {}).items():
                print(f"      • {feat}: {score:.1f}%")

        df_proc = df_res.copy()

        if nombre_config == "METROPOLITANA" or region_key == "RM":
            df_proc = df_proc[df_proc['Region_Norm'].isin(["METROPOLITANA", "RM"])].copy()
            
            mask_merge = df_proc['cluster_final'].isin([0, 1])
            count_merge = mask_merge.sum()
            if count_merge > 0:
                print(f"►► RM Merge: Fusionando {count_merge} faenas de Cluster 0 y 1 en un solo Cluster 0.")
                df_proc.loc[mask_merge, 'cluster_final'] = 0

            print("Rescatando ruido en RM (Percentil 30)...")
            df_proc = rescue_noise_geo(df_proc, matrix_times=times_region, percentil_rescate=20)
            prefijo = "RM"
        else:
            print(f"Rescatando ruido (Percentil {30 if region_key in ['I','II','III','XV'] else 20})...")
            df_proc = rescue_noise_geo(
                df_proc, 
                matrix_times=times_region, 
                percentil_rescate=30 if region_key in ["I","II","III","XV"] else 20
            )
            prefijo = region_key if len(region_key)<=3 else region_item[:3]

        # Propagar Silhouette_Cluster_Avg a puntos rescatados del ruido (basado en cluster_final_v2)
        if metadata and metadata.get('Per_Cluster_Silhouette'):
            cluster_sil = metadata['Per_Cluster_Silhouette']
            df_proc['Silhouette_Cluster_Avg'] = df_proc['cluster_final_v2'].apply(
                lambda x: cluster_sil.get(int(x), np.nan) if int(x) != -1 else np.nan
            )

        n_iniciales = (df_proc['cluster_final'] != -1).sum()
        n_finales = (df_proc['cluster_final_v2'] != -1).sum()
        n_rescatadas = n_finales - n_iniciales
        
        print(f"   ►► DETALLE ASIGNACIÓN:")
        print(f"      ► Asignadas por Cluster (HDBSCAN): {n_iniciales}")
        print(f"      ► Rescatadas del Ruido (Geo):      {n_rescatadas}")
        print(f"      ► Total Faenas Clusterizadas:      {n_finales}")

        df_proc['Cluster_ID'] = df_proc.apply(
            lambda x: f"{prefijo}-{x['cluster_final_v2']}" if x['cluster_final_v2'] != -1 else "Ruido",
            axis=1
        )
        
        final_cluster_count = df_proc[df_proc['cluster_final_v2'] != -1]['cluster_final_v2'].nunique()
        print(f"      ► Cantidad de Clusters Finales:   {final_cluster_count}")

        resultados_regionales.append(df_proc)
        print(f"►► {nombre_config} procesada. Total minas: {len(df_proc)}")

drop_cols = [
    'Empresa_MINERA CENTINELA',
    'Empresa_MINERA ESCONDIDA LTDA.',
    'Empresa_MINERA LAS CENIZAS S.A.',
    'Empresa_MINERA SPENCE  S.A.',
    'Empresa_MINERA LOS PELAMBRES',
    'Empresa_CIA MRA. TECK CARMEN DE ANDACOLLO',
    'Empresa_MINERA BMR SPA',
    'Empresa_EL ESPINO SPA',
    'Empresa_ANGLO AMERICAN SUR S.A.',
    'Empresa_MINERA CEMIN-PULLALLI SPA',
    'Empresa_SOC. EXPLORACION Y DESARROLLO MINERO (EXPLODESA)',
    'Empresa_AURYN MINING CHILE SPA',
    'Empresa_MINERA SAN PEDRO S.A.',
    'Empresa_MINERÍA Y CONSTRUCCIÓN HAROLD MOLINA TAPIA E.I.R.L',
    'Empresa_SOCIEDAD SERVICIOS RG MINERALS SPA',
    'Empresa_MINERA ALVAREZ II SPA.',
    'Empresa_JONATHAN CUSTODIO ADASME ADASME',
    'Empresa_JUAN CARLOS FERNANDEZ SALAZAR',
    'Empresa_SANTIAGO HUMBERTO MONTENEGRO VASQUEZ',
    'Empresa_SLOGGERS MINING SPA',
    'Empresa_JUAN AUGUSTO MUNIZAGA ROMAN',
    'Empresa_ANDRES BELISARIO FELIU ESCUDERO',
    'Empresa_EXTRACCIÓN MINERA ARTESANAL OLIVARES SPA',
    'Empresa_MINERA VALLE CENTRAL S.A.',
    'Empresa_PAMPA CAMARONES SpA',
    'Empresa_YANMAFER SPA',
    'Empresa_MIGUEL ANACONA ANACONA',
    'Empresa_Sociedad Minera Santa Beatriz SpA',
    'Empresa_APOLO MINERIA SpA',
    'Empresa_CIA. MINERA DOÑA INES DE COLLAHUASI S.C.M.',
    'Empresa_CÍA. MRA. TECK QUEBRADA BLANCA S. A.',
    'Empresa_INVERSIONES GENOVA S.P.A.',
    'Empresa_CORPORACION NACIONAL DEL COBRE (CODELCO)',
    'Empresa_SOCIEDAD PUNTA DEL COBRE S.A.',
    'Empresa_MANTOVERDE S.A.',
    'Empresa_EMPRESA NACIONAL DE MINERIA (ENAMI)',
    'Empresa_MINERA LUMINA COPPER CHILE LTDA',
    # Distance and time columns to cities
    'dist_Santiago_km', 'time_Santiago_min',
    'dist_Valparaíso_km', 'time_Valparaíso_min',
    'dist_Viña_del_Mar_km', 'time_Viña_del_Mar_min',
    'dist_Antofagasta_km', 'time_Antofagasta_min',
    'dist_La_Serena_km', 'time_La_Serena_min',
    'dist_Coquimbo_km', 'time_Coquimbo_min',
    'dist_Rancagua_km', 'time_Rancagua_min',
    'dist_Talca_km', 'time_Talca_min',
    'dist_Iquique_km', 'time_Iquique_min',
    'dist_Arica_km', 'time_Arica_min',
    'dist_Calama_km', 'time_Calama_min',
    'dist_Copiapó_km', 'time_Copiapó_min',
    'dist_San_Bernardo_km', 'time_San_Bernardo_min',
    'dist_Curicó_km', 'time_Curicó_min',
    'dist_Ovalle_km', 'time_Ovalle_min',
    # Port time columns
    'Tiempo_Prt_Antofagasta', 'Tiempo_Prt_Bahia Agua Fresca',
    'Tiempo_Prt_Bahia De Valdivia', 'Tiempo_Prt_Bahia De Valparaiso',
    'Tiempo_Prt_Bahia Harris', 'Tiempo_Prt_Bahia Herradura Guayacan',
    'Tiempo_Prt_Bahia Quintero (Ventanas)', 'Tiempo_Prt_Bahia San Vicente',
    'Tiempo_Prt_Caleta Clarencia', 'Tiempo_Prt_Caleta Mina Elena',
    'Tiempo_Prt_Caleta Patillos', 'Tiempo_Prt_Coquimbo',
    'Tiempo_Prt_Coronel', 'Tiempo_Prt_Huasco',
    'Tiempo_Prt_Iquique', 'Tiempo_Prt_Lirquen',
    'Tiempo_Prt_Lota', 'Tiempo_Prt_Mejillones',
    'Tiempo_Prt_Penco', 'Tiempo_Prt_Port San Juan De La Juan',
    'Tiempo_Prt_Puerto Caldera', 'Tiempo_Prt_Puerto Calderilla',
    'Tiempo_Prt_Puerto Castro', 'Tiempo_Prt_Puerto Chacabuco',
    'Tiempo_Prt_Puerto Chanaral', 'Tiempo_Prt_Puerto De Corral',
    'Tiempo_Prt_Puerto Montt', 'Tiempo_Prt_Puerto Natales',
    'Tiempo_Prt_Puerto Quemchi', 'Tiempo_Prt_Puerto San Antonio',
    'Tiempo_Prt_Puerto Sara', 'Tiempo_Prt_Puerto Williams',
    'Tiempo_Prt_Puerto Yartou', 'Tiempo_Prt_Punta Delgada',
    'Tiempo_Prt_Rada De Arica', 'Tiempo_Prt_Rada Punta Arenas',
    'Tiempo_Prt_Talcahuano', 'Tiempo_Prt_Taltal',
    'Tiempo_Prt_Tocopilla',
    # Clustering and region columns
    'Es_Estrategica'
]

# ------------------------------
# Guardar resultados
# ------------------------------
if resultados_regionales:
    df_final = pd.concat(resultados_regionales, ignore_index=True)
    
    df_final = df_final.drop(columns=[col for col in drop_cols if col in df_final.columns])
    
    # ------------------------------
    # MANUAL CLUSTER REASSIGNMENT
    # ------------------------------
    mask_lomas_bayas = (df_final['NombreEmpresa'] == 'COMPAÑÍA MINERA  LOMAS BAYAS') & (df_final['Cluster_ID'] == 'II-2')
    n_reassigned = mask_lomas_bayas.sum()
    
    if n_reassigned > 0:
        df_final.loc[mask_lomas_bayas, 'cluster_final_v2'] = 3
        df_final.loc[mask_lomas_bayas, 'Cluster_ID'] = 'II-3'
        print(f"\n►► REASIGNACIÓN MANUAL:")
        print(f"   ► {n_reassigned} faena(s) de COMPAÑÍA MINERA  LOMAS BAYAS movidas de II-2 a II-3")
    else:
        print(f"\n►► ADVERTENCIA: No se encontraron faenas de COMPAÑÍA MINERA  LOMAS BAYAS en cluster II-2 para reasignar")
    
    print(f"\n►► PROCESO FINALIZADO. Total Minas: {len(df_final)}")
    
    OUTPUT_FILE = os.path.join(BASE_DIR,"2_regional_davbou_hdbs.csv")
    df_final.to_csv(OUTPUT_FILE, index=False)
    print(f"► Resultados guardados en: {OUTPUT_FILE}")
else:
    print("\n►► No se generaron resultados.")


Buscando archivos cerca de: /Users/mac/TrabajoTesis/FinalResultsFolder/02_Clustering/notebooks ...
Processed folder found at: /Users/mac/TrabajoTesis/FinalResultsFolder/02_Clustering/outputs
INPUT_MATRICES: /Users/mac/TrabajoTesis/FinalResultsFolder/02_Clustering/outputs/matrices_chile_region.npz
INPUT_DATA: /Users/mac/TrabajoTesis/FinalResultsFolder/02_Clustering/outputs/1_clustering_input_ready.csv
►► Matriz NPZ cargada: /Users/mac/TrabajoTesis/FinalResultsFolder/02_Clustering/outputs/matrices_chile_region.npz. Regiones disponibles: ['I_matrix', 'I_ids', 'II_matrix', 'II_ids', 'III_matrix', 'III_ids', 'IV_matrix', 'IV_ids', 'RM_matrix', 'RM_ids', 'V_matrix', 'V_ids', 'VI_matrix', 'VI_ids', 'VII_matrix', 'VII_ids', 'XV_matrix', 'XV_ids']
►► Datos listos: 7930 faenas activas cargadas.

PROCESANDO: ATACAMA (Davies-Bouldin)
Iniciando Optuna...


►► Importancias CLUSTERING:
   ► Peso Matriz Vial (OSRM): 46.4%
   ► Peso Atributos (Data):   53.6%
   ► Davies-Bouldin Score:    0.3017  (↓ mejor)
   ► DBCV Score:              -0.0218  (↑ mejor, rango [-1, 1])
   ► Silhouette Score Global: 0.5809  (↑ mejor, rango [-1, 1])
   ► Variables más influyentes:
      • Cota: 9.5%
      • Longitud: 8.8%
      • Tiempo_Prt_Puerto Chanaral: 8.8%
      • Tiempo_Prt_Puerto Caldera: 8.6%
      • time_Antofagasta_min: 8.3%
Rescatando ruido (Percentil 30)...
   ►► DETALLE ASIGNACIÓN:
      ► Asignadas por Cluster (HDBSCAN): 1301
      ► Rescatadas del Ruido (Geo):      322
      ► Total Faenas Clusterizadas:      1623
      ► Cantidad de Clusters Finales:   4
►► ATACAMA procesada. Total minas: 2372

PROCESANDO: ANTOFAGASTA (Davies-Bouldin)
Iniciando Optuna...


►► Importancias CLUSTERING:
   ► Peso Matriz Vial (OSRM): 49.4%
   ► Peso Atributos (Data):   50.6%
   ► Davies-Bouldin Score:    0.3567  (↓ mejor)
   ► DBCV Score:              0.1398  (↑ mejor, rango [-1, 1])
   ► Silhouette Score Global: 0.6806  (↑ mejor, rango [-1, 1])
   ► Variables más influyentes:
      • time_Antofagasta_min: 9.4%
      • Tiempo_Prt_Mejillones: 9.1%
      • Tiempo_Prt_Bahia Agua Fresca: 8.0%
      • Tiempo_Prt_Taltal: 7.6%
      • time_Santiago_min: 7.5%
Rescatando ruido (Percentil 30)...
   ►► DETALLE ASIGNACIÓN:
      ► Asignadas por Cluster (HDBSCAN): 886
      ► Rescatadas del Ruido (Geo):      170
      ► Total Faenas Clusterizadas:      1056
      ► Cantidad de Clusters Finales:   7
►► ANTOFAGASTA procesada. Total minas: 1453

PROCESANDO: COQUIMBO (Davies-Bouldin)
Iniciando Optuna...


►► Importancias CLUSTERING:
   ► Peso Matriz Vial (OSRM): 47.0%
   ► Peso Atributos (Data):   53.0%
   ► Davies-Bouldin Score:    0.4731  (↓ mejor)
   ► DBCV Score:              -0.4220  (↑ mejor, rango [-1, 1])
   ► Silhouette Score Global: 0.5494  (↑ mejor, rango [-1, 1])
   ► Variables más influyentes:
      • Cota: 14.0%
      • time_Antofagasta_min: 10.7%
      • Longitud: 10.3%
      • time_Santiago_min: 10.0%
      • time_La_Serena_min: 10.0%
Rescatando ruido (Percentil 20)...
   ►► DETALLE ASIGNACIÓN:
      ► Asignadas por Cluster (HDBSCAN): 906
      ► Rescatadas del Ruido (Geo):      173
      ► Total Faenas Clusterizadas:      1079
      ► Cantidad de Clusters Finales:   3
►► COQUIMBO procesada. Total minas: 1768

PROCESANDO: VALPARAÍSO (Davies-Bouldin)
Iniciando Optuna...


►► Importancias CLUSTERING:
   ► Peso Matriz Vial (OSRM): 75.0%
   ► Peso Atributos (Data):   25.0%
   ► Davies-Bouldin Score:    0.3379  (↓ mejor)
   ► DBCV Score:              0.2681  (↑ mejor, rango [-1, 1])
   ► Silhouette Score Global: 0.5929  (↑ mejor, rango [-1, 1])
   ► Variables más influyentes:
      • Longitud: 9.5%
      • time_Santiago_min: 9.2%
      • time_Antofagasta_min: 9.0%
      • Tiempo_Prt_Puerto San Antonio: 8.9%
      • Latitud: 8.5%
Rescatando ruido (Percentil 20)...
   ►► DETALLE ASIGNACIÓN:
      ► Asignadas por Cluster (HDBSCAN): 1405
      ► Rescatadas del Ruido (Geo):      15
      ► Total Faenas Clusterizadas:      1420
      ► Cantidad de Clusters Finales:   2
►► VALPARAÍSO procesada. Total minas: 1480

PROCESANDO: METROPOLITANA DE SANTIAGO (Davies-Bouldin)
Iniciando Optuna...


►► Importancias CLUSTERING:
   ► Peso Matriz Vial (OSRM): 57.2%
   ► Peso Atributos (Data):   42.8%
   ► Davies-Bouldin Score:    0.1678  (↓ mejor)
   ► DBCV Score:              0.0000  (↑ mejor, rango [-1, 1])
   ► Silhouette Score Global: 0.8734  (↑ mejor, rango [-1, 1])
   ► Variables más influyentes:
      • Latitud: 70.7%
      • time_Santiago_min: 8.2%
      • time_Valparaíso_min: 6.4%
      • time_Antofagasta_min: 4.3%
      • Tiempo_Prt_Taltal: 2.9%
►► RM Merge: Fusionando 205 faenas de Cluster 0 y 1 en un solo Cluster 0.
Rescatando ruido en RM (Percentil 30)...
   ►► DETALLE ASIGNACIÓN:
      ► Asignadas por Cluster (HDBSCAN): 205
      ► Rescatadas del Ruido (Geo):      39
      ► Total Faenas Clusterizadas:      244
      ► Cantidad de Clusters Finales:   1
►► METROPOLITANA DE SANTIAGO procesada. Total minas: 400

PROCESANDO: O'HIGGINS (Davies-Bouldin)
Iniciando Optuna...


►► Importancias CLUSTERING:
   ► Peso Matriz Vial (OSRM): 60.2%
   ► Peso Atributos (Data):   39.8%
   ► Davies-Bouldin Score:    0.0849  (↓ mejor)
   ► DBCV Score:              0.2439  (↑ mejor, rango [-1, 1])
   ► Silhouette Score Global: 0.9123  (↑ mejor, rango [-1, 1])
   ► Variables más influyentes:
      • Longitud: 57.4%
      • Cota: 17.3%
      • Empresa_JUAN AUGUSTO MUNIZAGA ROMAN: 4.6%
      • time_Talca_min: 4.0%
      • time_Santiago_min: 4.0%
Rescatando ruido (Percentil 20)...
   ►► DETALLE ASIGNACIÓN:
      ► Asignadas por Cluster (HDBSCAN): 123
      ► Rescatadas del Ruido (Geo):      27
      ► Total Faenas Clusterizadas:      150
      ► Cantidad de Clusters Finales:   2
►► O'HIGGINS procesada. Total minas: 256

PROCESANDO: ARICA Y PARINACOTA (Davies-Bouldin)
Iniciando Optuna...


►► Importancias CLUSTERING:
   ► Peso Matriz Vial (OSRM): 50.0%
   ► Peso Atributos (Data):   50.0%
   ► Davies-Bouldin Score:    N/A  (↓ mejor)
   ► DBCV Score:              N/A  (↑ mejor, rango [-1, 1])
   ► Silhouette Score Global: N/A  (↑ mejor, rango [-1, 1])
   ► Variables más influyentes:
      • Cota: 46.9%
      • Longitud: 18.1%
      • time_Santiago_min: 16.9%
      • time_Arica_min: 8.9%
      • Latitud: 5.9%
Rescatando ruido (Percentil 30)...
   ►► DETALLE ASIGNACIÓN:
      ► Asignadas por Cluster (HDBSCAN): 40
      ► Rescatadas del Ruido (Geo):      3
      ► Total Faenas Clusterizadas:      43
      ► Cantidad de Clusters Finales:   1
►► ARICA Y PARINACOTA procesada. Total minas: 50

PROCESANDO: MAULE (Davies-Bouldin)
Iniciando Optuna...
►► No se pudo procesar MAULE

PROCESANDO: TARAPACÁ (Davies-Bouldin)
Iniciando Optuna...


►► Importancias CLUSTERING:
   ► Peso Matriz Vial (OSRM): 75.0%
   ► Peso Atributos (Data):   25.0%
   ► Davies-Bouldin Score:    0.1687  (↓ mejor)
   ► DBCV Score:              0.3839  (↑ mejor, rango [-1, 1])
   ► Silhouette Score Global: 0.8658  (↑ mejor, rango [-1, 1])
   ► Variables más influyentes:
      • time_Arica_min: 12.4%
      • time_Iquique_min: 9.5%
      • Tiempo_Prt_Mejillones: 7.8%
      • Tiempo_Prt_Tocopilla: 7.6%
      • time_Santiago_min: 7.6%
Rescatando ruido (Percentil 30)...
   ►► DETALLE ASIGNACIÓN:
      ► Asignadas por Cluster (HDBSCAN): 78
      ► Rescatadas del Ruido (Geo):      22
      ► Total Faenas Clusterizadas:      100
      ► Cantidad de Clusters Finales:   2
►► TARAPACÁ procesada. Total minas: 149

►► REASIGNACIÓN MANUAL:
   ► 20 faena(s) de COMPAÑÍA MINERA  LOMAS BAYAS movidas de II-2 a II-3

►► PROCESO FINALIZADO. Total Minas: 7928
► Resultados guardados en: /Users/mac/TrabajoTesis/FinalResultsFolder/02_Clustering/outputs/2_regional_davbou_hdbs.